# Converse2D v4: SRAM-resident fused kernels on A100

Choose **Runtime > Change runtime type > A100** first. Everything builds from `claude/v4-dev`; the checked
production extension and the research `flash_fft` extension take 15-25 minutes, the studies about 40 minutes:

1. `flash_study.py`: fused training (s1, s2, s3) vs production with the capacity switch (A100: 163 KB opt-in
   shared memory, so s1 100x100, s2 128x128 and s3 144x144 all run fused); gate, bitwise repeat, paired timing.
2. `seed_sweep.py`: the seeded gate over 8 seeds on the s1 and s2/s3 shapes (compensated backward).
3. `ops_study.py`: the USRNet data term (one 7x7 kernel per plane, nearest prior, s1/s2/s3) and the
   replicate/reflect/zero padding modes at s1; training gate + timing, then its 8-seed sweep.
4. `half_study.py`: the half-spectrum fused inference forward against production's inference path, with k given.
5. `s23_error_check.py`: 32 seeds x 8 s2/s3 cases (incl. the 128x128 and 144x144 planes and the data terms):
   fused, fused + device regularizer, and the transposed-production control against the FP64 reference.
6. `efficiency.py` (with Nsight Compute where permitted) and `error_anatomy.py` (now including the data terms
   and the device-regularizer path).

Send back the zip the last cell downloads.


In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
%cd /content
!rm -rf ConverseNet
!git clone --branch claude/v4-dev https://github.com/Yiozolm/ConverseNet.git
%cd /content/ConverseNet
!git log --oneline -3
!pip -q install ninja

In [ ]:
import os, time
os.environ.setdefault('MAX_JOBS', str(os.cpu_count() or 2))
OUT = f"/content/results/flash-{torch.cuda.get_device_name(0).replace(' ', '_')}-{time.strftime('%Y%m%dT%H%M%S')}"
!python -u tools/v4_flash_fft/flash_study.py --output {OUT} 2>&1 | tee /content/flash_study.log

In [ ]:
!python -u tools/v4_flash_fft/seed_sweep.py --output {OUT}/seed_sweep.json --cases circular_s1_b4_c128_96_pad2 forward_s1_b4_c128_100 forward_s2_b4_c64_64 forward_s2_b4_c64_32 forward_s3_b2_c32_48 forward_s3_b2_c32_24 2>&1 | grep -E 'seed|eligible|Error|geomean' | tee {OUT}/seed_sweep.log

In [ ]:
# Kernel batches (USRNet data term) and the s1 padding modes: training gate + timing, then the seeded gate.
!python -u tools/v4_flash_fft/ops_study.py --output {OUT}/ops 2>&1 | tee {OUT}/ops.log
!python -u tools/v4_flash_fft/ops_study.py --seeds 8 --output {OUT}/ops_seeds 2>&1 | grep -E 'seed|eligible|Error|geomean|\| ' | tee {OUT}/ops_seeds.log


In [ ]:
# Half-spectrum fused inference forward vs production's half-spectrum path (k and l given to every candidate).
!python -u tools/v4_flash_fft/half_study.py --output {OUT}/half 2>&1 | tee {OUT}/half.log


In [ ]:
# s2/s3 grad_weight / grad_bias: fused, fused + device regularizer, and the transposed-production control, 32 seeds.
!python -u tools/v4_flash_fft/s23_error_check.py --seeds 32 --output {OUT}/s23_check --cases forward_s2_b4_c64_32 forward_s2_b4_c64_48 forward_s2_b4_c64_64 forward_s3_b2_c32_32 forward_s3_b2_c32_24 forward_s3_b2_c32_48 data_s2_b4_c64_48_k7 data_s3_b4_c64_32_k7 2>&1 | grep -E 'regularizer|eligible|Error|\| ' | tee {OUT}/s23_check.log


In [ ]:
# Memory and FLOP/bandwidth efficiency; ncu errors (e.g. counter permissions) are recorded per case.
!python -u tools/v4_flash_fft/efficiency.py --output {OUT}/efficiency --ncu 2>&1 | grep -vE '==PROF==' | tee {OUT}/efficiency.log

In [ ]:
# Error anatomy of grad_bias / grad_weight (spectrum vs arithmetic vs shared), all planes plus the data terms,
# with the device-regularizer path.
!python -u tools/v4_flash_fft/error_anatomy.py --output {OUT}/error_anatomy.json --cases circular_s1_b4_c64_96_pad2 circular_s1_b4_c128_96_pad2 forward_s1_b4_c128_100 forward_s1_b4_c64_96 forward_s2_b4_c64_32 forward_s2_b4_c64_48 forward_s3_b2_c32_32 forward_s3_b2_c32_24 forward_s2_b4_c64_64 forward_s3_b2_c32_48 data_s2_b4_c64_48_k7 data_s3_b4_c64_32_k7 2>&1 | grep -vE 'Warning|warn' | tee {OUT}/error_anatomy.log


In [ ]:
from IPython.display import Markdown, display
for path in ('summary.md', 'ops/summary.md', 'ops_seeds/summary.md', 'half/summary.md', 's23_check/summary.md',
             'efficiency/efficiency.md'):
    try:
        display(Markdown(open(f'{OUT}/{path}').read()))
    except FileNotFoundError as error:
        print('missing:', error)


Download the results and send back the whole zip (every summary.md, the JSON files and the logs).


In [ ]:
import shutil
shutil.copy('/content/flash_study.log', OUT)
archive = shutil.make_archive(OUT, 'zip', OUT)
from google.colab import files
files.download(archive)